# Notebook 13 – Feature Engineering Pipeline

## 1. Feature Engineering Workflow

A feature engineering workflow is a structured process for preparing and creating features before giving data to a Machine Learning model.

A typical workflow is:

Raw Data
↓
Train/Test Separation
↓
Feature Creation
↓
Numerical Transformation
↓
Categorical Transformation
↓
Feature Selection
↓
ML-Ready Features

Using a pipeline makes this process reusable, consistent, and safer.

In [1]:
import pandas as pd

df = pd.read_csv("Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Train/Test Separation

The dataset should be separated into training and testing data before learning preprocessing parameters.

The training data is used to learn transformations, while the test data is only transformed using the information learned from the training data.

This helps prevent data leakage and gives a more realistic evaluation of the model.

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop("Survived", axis=1)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((712, 11), (179, 11))

## 3. Feature Creation

Feature creation means creating new features from existing columns to represent useful information more clearly.

For the Titanic dataset, FamilySize can be created using SibSp and Parch.

FamilySize = SibSp + Parch + 1

This can help the model understand the passenger's family size.

In [4]:
def create_features(data):
    data = data.copy()
    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
    return data

X_train = create_features(X_train)
X_test = create_features(X_test)

X_train[["SibSp", "Parch", "FamilySize", "IsAlone"]].head()

,SibSp,Parch,FamilySize,IsAlone
692,0,0,1,1
481,0,0,1,1
527,0,0,1,1
855,0,1,2,0
801,1,1,3,0


## 4. Numerical Transformations

Numerical features may contain missing values or different ranges.

A numerical pipeline can:
- Fill missing values.
- Scale numerical features.

Here, median imputation is used for missing numerical values and StandardScaler is used to standardize the values.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical pipeline created successfully.")
print("Features:", numeric_features)

Numerical pipeline created successfully.
Features: ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']


## 5. Categorical Transformations

Categorical features contain values such as Male/Female or different port categories.

A categorical pipeline can:
- Fill missing categories.
- Convert categories into numerical values.

One-Hot Encoding is suitable for nominal categorical features such as Sex and Embarked.

In [11]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["Sex", "Embarked"]

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

categorical_output = categorical_pipeline.fit_transform(
    X_train[categorical_features]
)

print("Categorical pipeline completed")
print("Output shape:", categorical_output.shape)

categorical_output[:5]

Categorical pipeline completed
Output shape: (712, 5)


array([[0., 1., 0., 0., 1.],
       [0., 1., 0., 0., 1.],
       [0., 1., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.]])

## 6. ColumnTransformer

ColumnTransformer allows different preprocessing steps to be applied to different groups of columns.

For example:
- Numerical columns → Imputation + Scaling
- Categorical columns → Imputation + One-Hot Encoding

This keeps all preprocessing steps organized in one structure.

In [12]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

X_train_ready.shape

(712, 10)

## 7. Transformation Pipeline

A transformation pipeline combines multiple preprocessing steps into a single reusable workflow.

The main advantage is that the same transformations are automatically applied in the correct order.

The pipeline also helps prevent inconsistent preprocessing between training and testing data.

In [13]:
from sklearn.linear_model import LogisticRegression

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

model_pipeline.fit(X_train, y_train)

accuracy = model_pipeline.score(X_test, y_test)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.7988826815642458


## 8. Feature Selection

Feature selection means selecting the most useful features and removing irrelevant or redundant features.

It can help:
- Reduce unnecessary features.
- Simplify the model.
- Reduce computational cost.
- Improve interpretability.

Feature selection should be performed using training data only so that test information does not influence the process.

In [14]:
selected_features = [
    "Pclass", "Age", "SibSp", "Parch",
    "Fare", "FamilySize", "IsAlone"
]

print("Selected Features:")
print(selected_features)

Selected Features:
['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']


## 9. Preventing Data Leakage

A pipeline helps prevent data leakage by ensuring that preprocessing steps are fitted only on the training data.

For example:

Training data → fit + transform
Testing data → transform only

We should never fit the scaler, encoder, or imputer separately on the test data.

This ensures that information from the test set does not influence the training process.

In [15]:
# Correct approach

preprocessor.fit(X_train)

X_train_ready = preprocessor.transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print("Training shape:", X_train_ready.shape)
print("Testing shape:", X_test_ready.shape)

Training shape: (712, 10)
Testing shape: (179, 10)


## 10. Required Documentation for Important Engineered Features

For every important engineered feature, document:

### Feature Name
Name of the newly created feature.

### Source Columns
Which columns were used?

### Logic
Explain the formula or business rule.

### Reason
Why was the feature created?

### Business Meaning
What does the feature represent?

### ML Relevance
How could this feature help a Machine Learning model?

### Leakage Check
Could this feature introduce data leakage?

### Final Decision
Choose one:
- Retain
- Remove
- Needs Further Analysis

In [16]:
documentation = pd.DataFrame([
    ["FamilySize", "SibSp, Parch", "SibSp + Parch + 1",
     "Represent family size", "Total family members",
     "May capture family-related patterns",
     "No target information used", "Retain"],

    ["IsAlone", "FamilySize", "FamilySize == 1",
     "Identify passengers travelling alone", "Travelling alone or not",
     "May capture survival differences",
     "No target information used", "Retain"]
], columns=[
    "Feature Name", "Source Columns", "Logic", "Reason",
    "Business Meaning", "ML Relevance",
    "Leakage Check", "Final Decision"
])

documentation

,Feature Name,Source Columns,Logic,Reason,Business Meaning,ML Relevance,Leakage Check,Final Decision
0,FamilySize,"SibSp, Parch",SibSp + Parch + 1,Represent family size,Total family members,May capture family-related patterns,No target information used,Retain
1,IsAlone,FamilySize,FamilySize == 1,Identify passengers travelling alone,Travelling alone or not,May capture survival differences,No target information used,Retain


## 11. Conclusion

A Feature Engineering Pipeline organizes feature creation and preprocessing into a reusable workflow.

In this notebook, we used:
- Train/Test Separation
- Feature Creation
- Numerical Transformations
- Categorical Transformations
- ColumnTransformer
- Pipeline
- Feature Selection
- Leakage Prevention

The main benefit is that the same preprocessing logic can be applied consistently to training and unseen data while reducing the risk of data leakage.

In [17]:
print("Feature Engineering Pipeline completed successfully.")
print("Train/Test leakage prevention applied.")

Feature Engineering Pipeline completed successfully.
Train/Test leakage prevention applied.
